In [23]:
pip install selenium pandas pulp nba_api scikit-learn

In [24]:
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (2,616 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
chromium-browser is already the newest version (2:1snap1-0ubuntu2).
chromium-chromedriver is already the newest version (2:1sn

In [25]:
# ---------- Dunkest NBA Optimizer – NBA_API + ML + 10 GIOCATORI ----------
# Requisiti: selenium, pandas, pulp, nba_api, scikit-learn
# pip install selenium pandas pulp nba_api scikit-learn

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import pulp
import time
import re
from datetime import date, timedelta
from nba_api.stats.endpoints import leaguedashplayerstats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
BUDGET = 107.2
MIN_CR = 4.0
HEADLESS = True
SLEEP_SCROLL = 0.2
MAX_PAGES = 300
malus_change = 5
NEWS_SECTION = "https://www.dunkest.com/it/nba/notizie/fantabasket"

# ---------- DRIVER ----------
def make_driver(headless=HEADLESS):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--blink-settings=imagesEnabled=false")
    service = Service('/usr/bin/chromedriver')
    return webdriver.Chrome(service=service, options=opts)

# ---------- NORMALIZE POSITION ----------
def normalize_position(pos_str):
    s = str(pos_str).upper()
    if 'C' in s and not ('G' in s or 'F' in s): return 'C'
    if any(k in s for k in ['PG','SG','G','GUARD']): return 'G'
    if any(k in s for k in ['SF','PF','F','FORWARD']): return 'F'
    if s and s[0] in ['G','F','C']: return s[0]
    return 'F'

# ---------- SCRAPING INFORTUNI ----------
def scrape_injured_players_by_team():
    driver = make_driver()
    driver.get("https://www.dunkest.com/it/nba/infortuni")
    time.sleep(1)
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SLEEP_SCROLL)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height: break
        last_height = new_height
    injured_players = []
    tables = pd.read_html(driver.page_source)
    for df in tables:
        if 'Giocatore' in df.columns:
            injured_players.extend(df['Giocatore'].dropna().tolist())
    driver.quit()
    return list(set([p.strip() for p in injured_players]))

# ---------- TROVA COACH ----------
def find_coach_from_article(giornata: int, headless=HEADLESS):
    driver = make_driver(headless=headless)
    driver.get(NEWS_SECTION)
    time.sleep(2)
    target_href = None
    max_scrolls = 20
    scroll_count = 0
    while scroll_count < max_scrolls and target_href is None:
        link_elements = driver.find_elements(By.XPATH, "//a[contains(@href,'/nba/notizie/')]")
        for el in link_elements:
            title = el.text.strip()
            href = el.get_attribute('href')
            if not href or not title:
                continue
            low = title.lower()
            if f"g{giornata}" in low and "coach" in low:
                target_href = href
                break
        if target_href:
            break
        try:
            btn = driver.find_element(By.XPATH, "//button[contains(text(),'Carica altre news')]")
            if btn.is_enabled():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1)
        except:
            break
        scroll_count += 1
    if not target_href:
        driver.quit()
        return None
    try:
        driver.get(target_href)
        time.sleep(1)
        art_text = driver.find_element(By.TAG_NAME, "body").text
    except:
        driver.quit()
        return None
    driver.quit()
    matches = re.findall(
        r"([A-Z][A-Za-z\.\'\-]+?\s[A-Z][A-Za-z\.\'\-]+?)\s*\([^\]*?,\s*([0-9]+(?:[,\.][0-9]+)?)\s*cr\)",
        art_text
    )
    if not matches: return None
    coach_name, coach_CR = min(matches, key=lambda x: float(x[1].replace(",", ".")))
    return {
        'coach_name': coach_name,
        'coach_CR': float(str(coach_CR).replace(",", ".")),
        'url': target_href
    }

# ---------- OPTIMIZER ----------
def optimize_lineup(df, budget_players, current_lineup=None, malus_change=5):
    n = len(df)
    if n == 0: return pd.DataFrame(), 0
    current_set = set(p.strip().lower() for p in current_lineup) if current_lineup else set()
    x = [pulp.LpVariable(f"x_{i}", cat="Binary") for i in range(n)]
    starter = [pulp.LpVariable(f"starter_{i}", cat="Binary") for i in range(n)]
    sixth = [pulp.LpVariable(f"sixth_{i}", cat="Binary") for i in range(n)]
    bench = [pulp.LpVariable(f"bench_{i}", cat="Binary") for i in range(n)]
    captain = [pulp.LpVariable(f"captain_{i}", cat="Binary") for i in range(n)]
    prob = pulp.LpProblem("Dunkest_Optimizer", pulp.LpMaximize)
    for i in range(n):
        prob += starter[i] + sixth[i] + bench[i] == x[i]
        prob += captain[i] <= starter[i]
    prob += pulp.lpSum(x) == 10
    prob += pulp.lpSum(x[i] for i in range(n) if df.loc[i,'role']=='G') == 4
    prob += pulp.lpSum(x[i] for i in range(n) if df.loc[i,'role']=='F') == 4
    prob += pulp.lpSum(x[i] for i in range(n) if df.loc[i,'role']=='C') == 2
    prob += pulp.lpSum(starter) == 5
    prob += pulp.lpSum(sixth) == 1
    prob += pulp.lpSum(bench) == 4
    prob += pulp.lpSum(captain) == 1
    prob += pulp.lpSum(x[i]*df.loc[i,'cost'] for i in range(n)) <= budget_players
    points = []
    for i in range(n):
        mult = 1.0*starter[i] + 1.0*sixth[i] + 0.5*bench[i] + 0.5*captain[i]
        malus = malus_change if current_set and df.loc[i,'Giocatore'].strip().lower() not in current_set else 0
        points.append(mult*df.loc[i,'expected_points'] - malus*x[i])
    prob += pulp.lpSum(points)
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[prob.status] != "Optimal": return pd.DataFrame(), 0
    sel_indices = [i for i in range(n) if x[i].value() == 1]
    sel_players = df.loc[sel_indices].copy()
    status = []
    for i in sel_indices:
        if starter[i].value() == 1:
            status.append("⭐ Capitano" if captain[i].value()==1 else "Titolare")
        elif sixth[i].value() == 1:
            status.append("Sesto Uomo")
        else:
            status.append("Panchina")
    sel_players["Status"] = status
    total_points_val = 0
    for i in sel_indices:
        base = df.loc[i,'expected_points']
        if starter[i].value() == 1:
            contrib = 1.5*base if captain[i].value()==1 else 1.0*base
        elif sixth[i].value() == 1:
            contrib = 1.0*base
        else:
            contrib = 0.5*base
        malus = malus_change if current_set and df.loc[i,'Giocatore'].strip().lower() not in current_set else 0
        total_points_val += contrib - malus
    return sel_players, total_points_val

# ---------- EXPECTED POINTS NBA_API ----------
def compute_expected_points_nba_api():
    """
    Ottiene statistiche NBA aggiornate usando nba_api e prevede PDK con RandomForest.
    Restituisce df con colonne: Giocatore, Squadra, role, cost, expected_points
    """
    df_players = leaguedashplayerstats.LeagueDashPlayerStats(season='2025-26', per_mode_detailed='PerGame').get_data_frames()[0]
    cols_needed = ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP', 'MIN', 'PTS', 'AST', 'REB', 'STL', 'BLK', 'TO']
    df_ml = df_players[cols_needed].copy()
    df_ml = df_ml.rename(columns={'PLAYER_NAME':'Giocatore','TEAM_ABBREVIATION':'Squadra','PTS':'PDK'})
    feature_cols = ['GP','MIN','AST','REB','STL','BLK','TO']
    X = df_ml[feature_cols]
    y = df_ml['PDK']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    df_ml['expected_points'] = model.predict(X)
    df_ml['cost'] = 10.0  # placeholder CR
    df_ml['role'] = 'F'    # placeholder ruolo
    return df_ml[['Giocatore','Squadra','role','cost','expected_points']]

# ---------- DISPLAY ----------
def display_final_lineup(sel_players, sel_coach, total_cost, total_points):
    df_out=sel_players.copy()
    df_out=df_out.rename(columns={'role':'Ruolo','cost':'CR','expected_points':'Expected Points'})
    for col in ['Giocatore','Ruolo','Squadra','CR','Expected Points','Status']:
        if col not in df_out.columns: df_out[col]=''
    df_out['CR']=df_out['CR'].apply(lambda x: round(float(x),1) if x!='' else '')
    df_out['Expected Points']=df_out['Expected Points'].apply(lambda x: round(float(x),2) if x!='' else '')
    df_out=df_out[['Giocatore','Ruolo','Squadra','CR','Expected Points','Status']]
    role_colors={'G':'#D0E8F2','F':'#FCE1E4','C':'#E1F5C4'}
    def color_row(x): return [f'background-color: {role_colors.get(x["Ruolo"],"white")}' if col=='Ruolo' else '' for col in x.index]
    df_styled=df_out.style.apply(color_row, axis=1).set_properties(**{'text-align':'center'}).hide(axis='index')
    display(df_styled)
    coach_name = sel_coach.get('coach_name','N/A') if isinstance(sel_coach, dict) else 'N/A'
    print("\n━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"🏀 Coach: {coach_name}")
    print(f"💰 Budget speso: {total_cost:.1f} / {BUDGET} CR")
    print(f"📈 Expected total points: {total_points:.2f}")
    print("━━━━━━━━━━━━━━━━━━━━━━━")

# ---------- MAIN FUNCTION ----------
def run_optimizer_nba_api(giornata: int, current_lineup=None, malus_change=5):
    print(f"📰 Cercando coach consigliato per la Giornata {giornata}...")
    coach_info = find_coach_from_article(giornata)
    coach_cost = coach_info['coach_CR'] if coach_info else 0.0
    print(f"💰 Budget iniziale per i giocatori: {BUDGET-coach_cost:.1f} CR\n")
    print("📊 Raccolta dati giocatori da NBA_API e predizione expected points...")
    df_expected = compute_expected_points_nba_api()
    print("🤕 Filtrando giocatori infortunati...")
    injured_set = set([p.strip().lower() for p in scrape_injured_players_by_team()])
    df_expected = df_expected[~df_expected['Giocatore'].apply(lambda x: str(x).strip().lower() in injured_set)].reset_index(drop=True)
    print(f"✅ Giocatori disponibili: {len(df_expected)}\n")
    print("🚀 Avvio ottimizzazione lineup...")
    sel_players, total_points = optimize_lineup(df_expected, budget_players=BUDGET-coach_cost, current_lineup=current_lineup, malus_change=malus_change)
    total_cost = sel_players['cost'].sum() + coach_cost
    display_final_lineup(sel_players, coach_info, total_cost, total_points)
    return sel_players, total_cost, total_points

In [26]:
current_lineup = ["Nikola Jokic","Jalen Johnson", "Bennedict Mathurin", "Grayson Allen", "Andre Drummond", "Ajay Mitchell", "Trent Watford", "Isaiah Jackson", "Dominick Barlow", "Landry Shamet"]

In [27]:
run_optimizer_nba_api(giornata=35, current_lineup=current_lineup , malus_change=5)

📰 Cercando coach consigliato per la Giornata 35...


WebDriverException: Message: Service /usr/bin/chromedriver unexpectedly exited. Status code was: 1
